# Modelling: search and ranking on OTTO sessions

Follows `_doc/modeling_guide.md`, one section per step, decisions in `modeling/model_log.md`, every experiment in `modeling/experiments.csv`, one issue per step (umbrella #110). It starts from the EDA deliverables (`eda/eda_summary.md`, `eda/preprocessing_spec.md`). Outputs are not committed with the notebook (nbstripout), so every step carries its reasoning in text cells: the candidates considered, why the step exists, what the result means.

Conventions used throughout: unit = one query (a session prefix plus its 200 candidate items); primary metric NDCG@10 of the ranking produced inside the pool, ideal DCG from the full label set; folds are made over queries; seed 42; a difference smaller than the noise threshold of Step 0 is a tie and the simpler model wins.

In [ ]:
import importlib.metadata as md_
import json
import platform
import sys
import time
from pathlib import Path

import numpy as np
import polars as pl

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "notebooks"))
from model_cv import (
    F15,
    POOL,
    cv,
    experiment_rows,
    fit_score,
    folds,
    lgbm,
    linear,
    load_dev,
    ndcg10,
    pool_ceiling,
    summarise,
)
from model_data import DEV_TABLE, HOLDOUT_TABLE, build_holdout, holdout_fingerprint
from model_utils import SEED, cached, log_experiments, log_step, save_table
from sklearn.metrics import average_precision_score, roc_auc_score
from split import DAY_MS, START_MS

VERSIONS = {p: md_.version(p) for p in ("polars", "numpy", "scikit-learn", "lightgbm", "scipy")}
print("python", platform.python_version(), VERSIONS, "| seed", SEED)

## Step 0. Setup and reference points
*Kind: diagnostic. Issue #111.* Read the inputs, settle the holdout, rebuild the EDA preprocessing as one pipeline, and set the reference score, the noise threshold and the success criterion that every later step is judged against.

#### Why this step exists here
Every later number is only meaningful against a reference, a noise level and an untouched test set. This step also checks that the EDA pipeline can be rebuilt and reproduces the EDA baselines; if it could not, nothing built on it could be trusted.

#### Options considered
- **Holdout:** keep the EDA holdout (sessions starting on days 14-15), build a fresh window from unused days (16-17), or use a later window (days 21-22 or 26-27). The first was rejected because earlier issues scored five methods on it; the third because the weekly-arrival demonstrations of the MLOps phase need those windows.
- **Baselines:** the rule already in use (the pool order from reciprocal-rank fusion), random order, single-feature rankings (co-visitation rank, popularity rank), regularised logistic regression, LightGBM at defaults as a classifier and as a lambdarank ranker.
- **Noise floor:** five fold seeds x three folds on 10,000 queries; the guide's threshold is the larger of the fold standard deviation and the seed spread of the reference; the paired difference between options on the same folds is reported next to it.
- **Ceiling:** a perfect reordering of the pool, the benchmarks of the earlier phase (reference only, different queries), label noise from the EDA.

In [ ]:
# A. inputs from the EDA phase, and what is assumed where the guide asks for facts we were not given
required = ["eda/eda_summary.md", "eda/eda_log.md", "eda/preprocessing_spec.md", "eda/tables/step17_excluded_columns.csv", "eda/tables/step16_baselines.csv", "data/features/clean_dev.parquet"]
missing = [p for p in required if not (ROOT / p).exists()]
assert not missing, f"missing inputs: {missing}"
print("all EDA inputs present:", ", ".join(required))
assumptions = pl.DataFrame([
    {"item": "primary metric", "value": "NDCG@10 inside the 200-candidate pool, ideal DCG from the full label set, mean over queries", "status": "assumed (from AGENTS.md and issues #13/#17; the EDA used PR-AUC only for speed)"},
    {"item": "error costs", "value": "unknown; a ranking has no fixed cut-off, the operating point is the top-10 result list", "status": "assumed; no cost-based threshold is invented (Step 5)"},
    {"item": "latency limit", "value": "under 100 ms per query (200 candidates) on one Lambda CPU, p95", "status": "assumed (serverless architecture, low traffic)"},
    {"item": "model size limit", "value": "under 50 MB for the frozen artifact", "status": "assumed"},
    {"item": "interpretability", "value": "feature importance and a model card are enough; no per-decision explanation required", "status": "assumed"},
    {"item": "prohibited features", "value": "none; identifiers (session, aid) are not used as features (EDA Steps 6, 13)", "status": "assumed"},
    {"item": "compute budget", "value": "one 8 GB laptop CPU, no cloud spend, hours not days; screening on 10,000 queries, finalists on the 40,000", "status": "assumed"},
    {"item": "retraining frequency", "value": "weekly (matches the weekly data-arrival plan of the MLOps phase)", "status": "assumed"},
    {"item": "sensitive attributes", "value": "none in the data; disparity analysis in Step 6 is limited to the synthetic categories and prefix length", "status": "assumed"},
])
save_table("step0_assumptions", assumptions); print(assumptions.select("item", "value", "status"))

In [ ]:
# B. the holdout. The EDA said days 14-15 were never read by the EDA notebook; but earlier phases scored five methods on that window.
cmp = json.loads((ROOT / "data" / "results" / "comparison.json").read_text())
spent = [(r["method"], r["n"], r["ndcg10"]) for r in cmp["full_window"]]
print("window of days 14-15 (features as of day 14) was already used to compare methods in issues #17-#29:")
for m, n, v in spent: print(f"   {m}: NDCG@10 {v} on {n:,} queries")
print("-> it cannot be an untouched holdout (rule 1). A fresh window is built from days that no earlier issue used.")
fp = build_holdout()
fp2 = holdout_fingerprint(); assert fp["sha256_of_sorted_session_ids"] == fp2["sha256_of_sorted_session_ids"]
dev_sessions = set(pl.scan_parquet(DEV_TABLE).select("session").unique().collect()["session"].to_list())
w0_sessions = set(pl.scan_parquet(ROOT / "data" / "features" / "eval_w0.parquet").select("session").unique().collect()["session"].to_list())
h0_sessions = set(pl.scan_parquet(HOLDOUT_TABLE).select("session").unique().collect()["session"].to_list())
assert not (h0_sessions & dev_sessions) and not (h0_sessions & w0_sessions), "the fresh holdout shares sessions with development or the used window"
print("fresh holdout:", fp)
print(f"no session is shared with development ({len(dev_sessions):,} queries) or with the used window ({len(w0_sessions):,}); the holdout table is written but has not been scored")
dev_fp = {"queries": len(dev_sessions), "rows": len(dev_sessions) * POOL}

In [ ]:
# C. rebuild the EDA pipeline and reproduce the EDA baselines (Step 16 of the EDA: time-based split, 3 samples of 6,000 queries)
first = pl.scan_parquet(ROOT / "data" / "sample" / "events.parquet").group_by("session").agg(pl.col("ts").min().alias("first_ts")).collect()
def days_of(sessions):
    return pl.DataFrame({"session": sessions}).join(first, on="session", how="left")["first_ts"].to_numpy().__sub__(START_MS) // DAY_MS
def time_split(q):
    d = days_of(q.sessions); return np.flatnonzero(d <= 12), np.flatnonzero(d == 13)
rows = []
for s in (42, 43, 44):
    q = load_dev(6000, seed=s); tr, va = time_split(q); q_tr, q_va = q.take(tr), q.take(va)
    for spec in (linear(), lgbm()):
        sc, *_ = fit_score(spec, q_tr, q_va)
        rows.append({"option": spec.name, "sample": s, "pr_auc": average_precision_score(q_va.y, sc.ravel()), "ndcg10": float(ndcg10(sc, q_va.grade, q_va.idcg).mean())})
rep = pl.DataFrame(rows).group_by("option", maintain_order=True).agg(pl.col("pr_auc").mean(), pl.col("pr_auc").std().alias("pr_auc_sd"), pl.col("ndcg10").mean())
eda = pl.read_csv(ROOT / "eda" / "tables" / "step16_baselines.csv").filter(pl.col("option").str.starts_with("B.") | pl.col("option").str.starts_with("C."))
eda_pr, eda_sd = eda["pr_auc"].to_list(), eda["pr_auc_sd"].to_list()
rep = rep.with_columns(pl.Series("eda_pr_auc", eda_pr), pl.Series("eda_spread", eda_sd)).with_columns(((pl.col("pr_auc") - pl.col("eda_pr_auc")).abs() <= pl.max_horizontal("pr_auc_sd", "eda_spread")).alias("reproduced"))
save_table("step0_reproduce_eda", rep); print(rep)
assert rep["reproduced"].all(), "the rebuilt pipeline does not reproduce the EDA baselines: stop and find out why"

In [ ]:
# D. baselines on the screening sample (10,000 queries) with five fold seeds: the noise floor
dev10 = load_dev(10_000, seed=SEED); SEEDS5 = (42, 43, 44, 45, 46)
print(f"screening sample: {dev10.nq:,} queries, {dev10.df.height:,} rows, {int(dev10.y.sum()):,} positives")
def heuristic(name, fn):
    out = []
    for seed in SEEDS5:
        for fold, (_, va) in enumerate(folds(dev10.nq, seed)):
            qv = dev10.take(va); s = fn(qv)
            out.append({"option": name, "seed": seed, "fold": fold, "ndcg10": float(ndcg10(s, qv.grade, qv.idcg).mean()), "pr_auc": average_precision_score(qv.y, s.ravel()), "roc_auc": roc_auc_score(qv.y, s.ravel()),
                        "fit_seconds": 0.0, "predict_seconds": 0.0, "size_mb": 0.0, "n_features": 0, "queries": qv.nq})
    return out
pos = -np.arange(POOL, dtype=np.float64)[None, :]
HEUR = {"pool order (rule in use: reciprocal-rank fusion)": lambda q: np.repeat(pos, q.nq, 0),
        "random within the pool": lambda q: np.random.default_rng(SEED).random((q.nq, POOL)),
        "co-visitation rank": lambda q: -q.df["rank_covis"].to_numpy().reshape(q.nq, POOL) + 1e-6 * np.repeat(pos, q.nq, 0),
        "popularity rank": lambda q: -q.df["rank_pop"].to_numpy().reshape(q.nq, POOL) + 1e-6 * np.repeat(pos, q.nq, 0)}
def run0():
    rows = [r for n, fn in HEUR.items() for r in heuristic(n, fn)]
    for spec in (linear(), lgbm(), lgbm(name="LightGBM lambdarank defaults", rank=True)):
        t0 = time.time(); r, _ = cv(spec, dev10, seeds=SEEDS5); rows += r.to_dicts(); print(f"  {spec.name}: {time.time() - t0:.0f} s")
    return pl.DataFrame(rows)
cv0 = cached("step0_baselines_rows", run0); sum0 = summarise(cv0); save_table("step0_baselines", sum0)
print(sum0.select("option", "ndcg10", "ndcg10_sd", "seed_spread", "pr_auc", "roc_auc", "fit_seconds", "size_mb").sort("ndcg10", descending=True))

In [ ]:
# E. noise floor, ceiling, reference and success criterion
ref_name = sum0.sort("ndcg10", descending=True)["option"][0]; ref = sum0.filter(pl.col("option") == ref_name).row(0, named=True)
noise = max(ref["ndcg10_sd"], ref["seed_spread"])
# paired view: how much of the fold-to-fold variation is shared between options (same folds)?
pv = cv0.select("option", "seed", "fold", "ndcg10"); refv = pv.filter(pl.col("option") == ref_name).rename({"ndcg10": "ref"}).drop("option")
paired = pv.join(refv, on=["seed", "fold"]).with_columns((pl.col("ndcg10") - pl.col("ref")).alias("d")).group_by("option").agg(pl.col("d").mean().alias("mean_diff_to_reference"), pl.col("d").std().alias("paired_sd")).sort("mean_diff_to_reference", descending=True)
save_table("step0_paired_to_reference", paired); print("paired difference to the reference (same folds):"); print(paired)
ceil = pool_ceiling(dev10.grade, dev10.idcg); success = 2 * noise
share_zero = float((ceil == 0).mean())
print(f"reference: {ref_name} NDCG@10 {ref['ndcg10']:.4f}; fold sd {ref['ndcg10_sd']:.4f}, seed spread {ref['seed_spread']:.4f} -> noise threshold {noise:.4f}; success criterion (2 x noise) {success:.4f}")
print(f"ceiling: a perfect reordering of the 200-candidate pool reaches NDCG@10 {ceil.mean():.4f}; {share_zero:.1%} of queries have no relevant item in their pool (nothing to find)")
print("benchmarks on the spent window (reference only, different queries): co-visitation + fill 0.2144, LightGBM lambdarank 0.2153, DCN-V2 + MMoE 0.2069 (data/results/comparison.json)")
print("label noise: 14 of 8,000,000 rows have conflicting labels (EDA Step 4), negligible")
ceiling_tbl = pl.DataFrame([{"reference": ref_name, "ndcg10": ref["ndcg10"], "noise_threshold": noise, "success_criterion": success, "pool_ceiling": float(ceil.mean()), "queries_without_reachable_item": share_zero}])
save_table("step0_reference", ceiling_tbl)
desc = {n: n for n in sum0["option"].to_list()}
print(log_experiments(experiment_rows(0, cv0, sum0, desc, "15 EDA features" if False else "F15", {}, SEEDS5)), "experiments logged")

In [ ]:
top = sum0.sort("ndcg10", descending=True); best_learned = sum0.filter(pl.col("option").str.contains("regularised|LightGBM")).sort("ndcg10", descending=True).row(0, named=True)
e0 = log_step(0, "Setup and reference points", "Diagnostic",
    "Inputs and assumptions listed; holdout status checked and a fresh holdout built; EDA baselines reproduced with the rebuilt pipeline; baselines on the 10,000-query screening sample with five fold seeds: pool order (the rule already in use), random, co-visitation rank, popularity rank, regularised linear, LightGBM defaults (classifier, class weights), LightGBM lambdarank defaults; ceiling = perfect reordering of the pool.",
    f"Holdout: days 14-15 were already used to score five methods in issues #17-#29 (best NDCG@10 0.2153), so they are treated as spent; a fresh holdout of {fp['queries']:,} queries ({fp['rows']:,} rows; sessions starting on days 16-17, features as of day 16) was built and fingerprinted (sha256 of sorted session ids {fp['sha256_of_sorted_session_ids'][:12]}...), disjoint from development and from the used window, and not scored. Reproduction: the rebuilt pipeline gives PR-AUC " + ", ".join(f"{r['option']} {r['pr_auc']:.4f} (EDA {r['eda_pr_auc']:.4f})" for r in rep.iter_rows(named=True)) + ". Baseline NDCG@10 on the screening sample (mean over 15 folds): " + "; ".join(f"{r['option']} {r['ndcg10']:.4f}" for r in top.iter_rows(named=True)) + f". Reference: {ref_name} {ref['ndcg10']:.4f}; fold sd {ref['ndcg10_sd']:.4f}, seed spread {ref['seed_spread']:.4f}, noise threshold {noise:.4f}, success criterion {success:.4f}. Paired difference to the reference on the same folds: " + "; ".join(f"{r['option']} {r['mean_diff_to_reference']:+.4f} (paired sd {r['paired_sd']:.4f})" for r in paired.iter_rows(named=True) if r["option"] != ref_name) + f". Ceiling: {ceil.mean():.4f} (a perfect pool reordering); {share_zero:.0%} of queries have no relevant item in the pool. No learned baseline beats the reference: the reference is one feature (the co-visitation rank).",
    f"Reference score = {ref_name} at {ref['ndcg10']:.4f}. Noise threshold {noise:.4f} (larger of fold standard deviation and seed spread of the reference). Success criterion {success:.4f} NDCG@10 over the reference (2 x noise; no business figure was given). Holdout for the modelling phase: the fresh window (days 16-17), scored once in Step 8; the days 14-15 window is spent and only a reference.",
    "Rule 1 of the guide (holdout used once) cannot hold for days 14-15, so a new window from unused days was built with the same code as issue #22. The noise threshold follows the guide (larger of fold sd and seed spread of the reference). The constraints the guide asked for were not given, so they are assumed and marked (table above).",
    "Keeping days 14-15 as the holdout: rejected, it was used to compare methods. Using a later window that the MLOps plan needs (days 21-22, 26-27): rejected, it would spend windows meant for the weekly-arrival demonstrations. Rebuilding the pipeline by a different route: not needed, the EDA baselines were reproduced within their spread.",
    "High for the holdout decision and the reproduction; Medium for the noise threshold (three folds of 3,333 queries; the paired differences are much tighter than this threshold, see the paired table).",
    "All later steps compare against the reference and the noise threshold; if nothing beats the reference by more than the noise threshold after Steps 1 to 3 the guide says to stop and ask the human. Step 8 scores the fresh holdout once. Step 6 must flag the query-category features (the query category comes from the held-out click, known leak of issue #10).")
print(e0)

#### Chosen approach and rationale
**Decision.** Reference score = co-visitation rank at 0.1815. Noise threshold 0.0082 (larger of fold standard deviation and seed spread of the reference). Success criterion 0.0164 NDCG@10 over the reference (2 x noise; no business figure was given). Holdout for the modelling phase: the fresh window (days 16-17), scored once in Step 8; the days 14-15 window is spent and only a reference.

**Why.** Rule 1 of the guide (holdout used once) cannot hold for days 14-15, so a new window from unused days was built with the same code as issue #22. The noise threshold follows the guide (larger of fold sd and seed spread of the reference). The constraints the guide asked for were not given, so they are assumed and marked (table above).

**Rejected.** Keeping days 14-15 as the holdout: rejected, it was used to compare methods. Using a later window that the MLOps plan needs (days 21-22, 26-27): rejected, it would spend windows meant for the weekly-arrival demonstrations. Rebuilding the pipeline by a different route: not needed, the EDA baselines were reproduced within their spread.

**Confidence.** High for the holdout decision and the reproduction; Medium for the noise threshold (three folds of 3,333 queries; the paired differences are much tighter than this threshold, see the paired table).

#### Interpretation: what we understand from the results
Three findings shape everything that follows. First, the EDA statement that the holdout was never read was true of the EDA notebook only: days 14-15 had been used to compare five methods in the first phase, so it was replaced by a fresh window (43,782 queries, days 16-17) that is fingerprinted and will be scored once in Step 8. Second, the rebuilt pipeline reproduces the EDA baselines (PR-AUC 0.0579 and 0.0592 against 0.0581 and 0.0597). Third, and most important: no learned baseline beats the simplest reference. Sorting the pool by the co-visitation rank alone gives NDCG@10 0.1815; regularised regression, LightGBM and lambdarank at defaults score 0.1798, 0.1799 and 0.1787 (paired differences -0.0018, -0.0017 and -0.0028, paired sd about 0.002). They do beat it on PR-AUC (0.060-0.065 against 0.057), so they order the whole pool better, but not the top ten, which is what NDCG@10 rewards; this is the same picture as the EDA (models add +0.008 to +0.010 PR-AUC over the co-visitation rank). The rule already in use, the fused pool order, scores only 0.128, because the popularity ranks it blends in are weak (popularity rank alone 0.040, random 0.0135). The guide's noise threshold is 0.0082, set by the fold-to-fold difficulty of three folds of 3,333 queries; the paired differences between options are five times tighter, so the threshold is conservative and is used for ties, while key comparisons also get a paired bootstrap interval. The ceiling is 0.566 for a perfect reordering, but 38% of queries have no relevant item in their pool at all: the pool, not the ranker, limits the score. The success criterion (twice the noise threshold) is 0.0164 over the reference; if Steps 1 to 3 cannot reach it the guide requires asking the human. One caveat carried forward: the query category comes from the held-out click (known leak, issue #10), so absolute scores overstate what real search would give.

#### Impact on next steps
All later steps compare against the reference and the noise threshold; if nothing beats the reference by more than the noise threshold after Steps 1 to 3 the guide says to stop and ask the human. Step 8 scores the fresh holdout once. Step 6 must flag the query-category features (the query category comes from the held-out click, known leak of issue #10).

#### Out of scope
- A business figure for the smallest useful uplift: none given, so twice the noise threshold is used (assumed).
- Raising the share of queries with a reachable item (38% have none): candidate generation, not modelling.
- Confirmation of the assumed constraints (latency, size, weekly retraining): listed for review in Step 9.

## Step 1. Features
*Kind: selection. Issue #112.* Eight candidate families, each with a one-line hypothesis, added one at a time to the current best set (the 15 EDA features) and scored with a linear model and LightGBM on the same folds (screening sample, two fold seeds x 3 folds). A family is kept if it lifts one model by more than the noise threshold of Step 0 and does not hurt the other by more than it. Then nine feature-selection options, all learned inside the training folds.

#### Why this step exists here
The EDA left 15 usable features. Before comparing models, we ask whether new features can lift the ranking and which of the existing ones are worth keeping, because every model in the slate will use the result. It follows Step 0 because a gain only counts if it exceeds the noise threshold and moves the score toward the reference.

#### Options considered
- **Families, each with a hypothesis:** ratios and interactions (association relative to popularity, recent versus all-time clicks); quantile bins for the linear model; query-context aggregates (percentile and z-score inside the 200-candidate pool, share of candidates with a co-visitation list); time parts (hour, weekday); category frequency; cross-fitted category target rate; structure (cluster distances, principal components); missingness (count and joint pattern of the two coded ranks).
- **Selection options, all learned inside the training folds:** keep all, drop near-constant, drop the EDA no-signal features, drop the EDA-redundant features, L1, stability selection, tree importance, permutation backward elimination, recursive elimination.
- **Rule:** keep a family only if it lifts one model by more than the noise threshold and hurts neither; choose the smallest set within noise of the best, confirmed on fresh fold seeds.

In [ ]:
from dataclasses import replace

from model_cv import Prep
from model_features import (
    Bins,
    CategoryFrequency,
    CategoryTargetRate,
    Missingness,
    QueryContext,
    Ratios,
    Structure,
    TimeParts,
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import RobustScaler

ref0 = pl.read_csv(ROOT / "modeling" / "tables" / "step0_reference.csv").row(0, named=True)
NOISE = ref0["noise_threshold"]; SUCCESS = ref0["success_criterion"]; REFERENCE = ref0["ndcg10"]
print(f"noise threshold {NOISE:.4f}, success criterion {SUCCESS:.4f}, reference {REFERENCE:.4f} ({ref0['reference']})")
SEEDS2 = (42, 43)
FAMILIES = {
    "ratios, differences, interactions": (Ratios(), "co-visitation strength matters relative to the item's popularity; recent vs all-time clicks is a trend"),
    "non-linear forms (bins)": (Bins(), "the linear model cannot bend; quantile bins of the strongest features let it follow the decile shapes of the EDA"),
    "query-context aggregates": (QueryContext(), "relevance is relative inside a pool: the percentile and z-score of a value among the 200 candidates, and the share of candidates with a co-visitation list, carry more than the raw value"),
    "time features (hour, weekday)": (TimeParts(first, START_MS), "browsing intent differs by hour and weekday"),
    "categorical: category frequency": (CategoryFrequency(), "category size may say something about intent (EDA expects no gain)"),
    "categorical: cross-fitted category target rate": (CategoryTargetRate(), "some synthetic categories are easier to hit than others (EDA: Cramer's V 0.015)"),
    "structure (cluster distances, principal components)": (Structure(), "the EDA found one small cluster with 2.4x the target rate; distances and leading components may expose it"),
    "missingness (count, joint pattern)": (Missingness(), "where a candidate came from (co-visitation list, popularity list or both) is a compact description of it"),
}
# register: source, availability and compute cost (measured on 1,000 queries) of every family
probe = dev10.take(np.arange(1000)); reg = []
for fam, (ex, hyp) in FAMILIES.items():
    ex.fit(probe); t0 = time.time(); names, a = ex(probe); ms = (time.time() - t0) * 1000
    reg.append({"family": fam, "features": len(names), "hypothesis": hyp, "source": "candidate rows of the query" + (" + request time" if "time" in fam else "") + (" + training queries (fit inside folds)" if fam.startswith(("categorical", "structure", "non-linear")) else ""),
                "available_at": "prediction time (pool and prefix are known before ranking)", "compute_ms_per_1000_queries": round(ms, 1)})
reg = pl.DataFrame(reg); save_table("step1_register", reg); print(reg.select("family", "features", "compute_ms_per_1000_queries", "hypothesis"))
print("features suggested by the human: none received (asked in the log)")

In [ ]:
# A. one family at a time, linear model and LightGBM, same folds as Step 0 (seeds 42 and 43)
BASE = {"regularised linear": linear(), "LightGBM defaults (class weights)": lgbm()}
def run1a():
    rows = []
    for fam, (ex, _) in FAMILIES.items():
        for bname, spec in BASE.items():
            t0 = time.time(); r, _ = cv(replace(spec, name=f"{bname} + {fam}", extras=(ex,)), dev10, seeds=SEEDS2)
            rows += r.with_columns(pl.lit(bname).alias("base_model"), pl.lit(fam).alias("family")).to_dicts()
        print(f"  {fam}: {time.time() - t0:.0f} s (last model)")
    return pl.DataFrame(rows)
cv1a = cached("step1_families_rows", run1a)
base_rows = cv0.filter(pl.col("option").is_in(list(BASE)) & pl.col("seed").is_in(SEEDS2)).select(pl.col("option").alias("base_model"), "seed", "fold", pl.col("ndcg10").alias("ndcg10_base"), pl.col("pr_auc").alias("pr_auc_base"))
pair1 = cv1a.join(base_rows, on=["base_model", "seed", "fold"]).with_columns((pl.col("ndcg10") - pl.col("ndcg10_base")).alias("d"), (pl.col("pr_auc") - pl.col("pr_auc_base")).alias("dpr"))
fam1 = (pair1.group_by("family", "base_model", maintain_order=True).agg(pl.col("ndcg10").mean(), pl.col("d").mean().alias("delta_ndcg10"), pl.col("d").std().alias("paired_sd"), (pl.col("d") > 0).sum().alias("folds_better"), pl.len().alias("folds"),
        pl.col("dpr").mean().alias("delta_pr_auc"), pl.col("n_features").first().alias("n_features"), pl.col("fit_seconds").mean().alias("fit_seconds"))
        .with_columns(pl.when(pl.col("delta_ndcg10") > NOISE).then(pl.lit("helps")).when(pl.col("delta_ndcg10") < -NOISE).then(pl.lit("hurts")).otherwise(pl.lit("within noise")).alias("verdict")))
save_table("step1_families", fam1); print(fam1.select("family", "base_model", "delta_ndcg10", "paired_sd", "folds_better", "delta_pr_auc", "verdict").sort("family", "base_model"))

In [ ]:
# keep rule: helps at least one model beyond the noise and hurts neither beyond it
keep_rule = fam1.group_by("family").agg((pl.col("verdict") == "helps").any().alias("helps_one"), (pl.col("verdict") == "hurts").any().alias("hurts_one"), pl.col("delta_ndcg10").max().alias("best_delta"), pl.col("delta_ndcg10").min().alias("worst_delta"))
keepers = keep_rule.filter(pl.col("helps_one") & ~pl.col("hurts_one"))["family"].to_list()
print("families kept:", keepers or "none"); print(keep_rule.sort("best_delta", descending=True))
nominal = fam1.sort("delta_ndcg10", descending=True).row(0, named=True)
print(f"largest nominal gain: {nominal['family']} on {nominal['base_model']}: {nominal['delta_ndcg10']:+.4f} NDCG@10 ({nominal['folds_better']} of {nominal['folds']} folds better), below the noise threshold {NOISE:.4f}" if not keepers else "")
CURRENT = ("F15" if not keepers else "F15 + " + " + ".join(keepers))

In [ ]:
# B. feature-selection options on the current best set, each learned inside the training fold
def base_of(n): return n.replace("_missing", "")
def surrogate(x, seed=0, n=300_000):
    idx = np.random.default_rng(seed).choice(len(x), min(len(x), n), replace=False)
    return RobustScaler().fit_transform(np.nan_to_num(x[idx], nan=0.0)), idx
def to_idx(names, keep_base): return [i for i, n in enumerate(names) if base_of(n) in keep_base]
def selector(fn):
    def sel(x, q, names, family): return to_idx(names, fn(x, q, [base_of(n) for n in dict.fromkeys(names)], names, family))
    return sel
def s_all(x, q, bases, names, fam): return set(bases)
def s_near_constant(x, q, bases, names, fam):
    keep = set()
    for j, n in enumerate(names):
        v = np.nan_to_num(x[:200_000, j], nan=-9.0); _, cnt = np.unique(np.round(v, 6), return_counts=True)
        if cnt.max() / len(v) < 0.999: keep.add(base_of(n))
    return keep
def s_eda_no_signal(x, q, bases, names, fam): return set(bases) - {"cart_rate", "prefix_n_cart", "prefix_n_order"}
def s_eda_redundant(x, q, bases, names, fam): return set(bases) - {"covis_mean", "covis_wsum"}
def _l1(x, y, C=0.02):
    m = LogisticRegression(penalty="l1", solver="liblinear", C=C, max_iter=200).fit(x, y); return np.abs(m.coef_[0]) > 1e-8
def s_l1(x, q, bases, names, fam):
    xs, idx = surrogate(x); nz = _l1(xs, q.y[idx]); return {base_of(n) for n, k in zip(names, nz, strict=True) if k}
def s_stability(x, q, bases, names, fam):
    freq = np.zeros(len(names))
    for s in range(10):
        xs, idx = surrogate(x, seed=s, n=100_000); freq += _l1(xs, q.y[idx])
    return {base_of(n) for n, f in zip(names, freq / 10, strict=True) if f >= 0.6}
def s_tree_importance(x, q, bases, names, fam):
    idx = np.random.default_rng(0).choice(len(x), min(len(x), 300_000), replace=False)
    m = lgbm().make().fit(x[idx], q.y[idx]); g = m.booster_.feature_importance("gain"); g = g / g.sum()
    return {base_of(n) for n, v in zip(names, g, strict=True) if v >= 0.02}
def s_perm_backward(x, q, bases, names, fam):
    idx = np.random.default_rng(0).choice(len(x), min(len(x), 300_000), replace=False); cut = int(len(idx) * 0.7); tr_i, va_i = idx[:cut], idx[cut:]
    active = list(range(len(names)))
    for _ in range(3):
        m = lgbm().make().fit(x[tr_i][:, active], q.y[tr_i]); base = average_precision_score(q.y[va_i], m.predict_proba(x[va_i][:, active])[:, 1]); drop = []
        for k, j in enumerate(active):
            xv = x[va_i][:, active].copy(); xv[:, k] = np.random.default_rng(1).permutation(xv[:, k])
            if base - average_precision_score(q.y[va_i], m.predict_proba(xv)[:, 1]) <= 0: drop.append(j)
        if not drop: break
        active = [j for j in active if j not in drop]
    return {base_of(names[j]) for j in active}
def s_rfe(x, q, bases, names, fam):
    idx = np.random.default_rng(0).choice(len(x), min(len(x), 300_000), replace=False); active = list(range(len(names)))
    while len(active) > 8:
        m = lgbm().make().fit(x[idx][:, active], q.y[idx]); g = m.booster_.feature_importance("gain"); worst = np.argsort(g)[:2]
        active = [j for k, j in enumerate(active) if k not in set(worst.tolist())]
    return {base_of(names[j]) for j in active}
SELECTORS = {"keep all": (s_all, 1), "drop near-constant": (s_near_constant, 2), "drop EDA no-signal (cart_rate, prefix_n_cart, prefix_n_order)": (s_eda_no_signal, 2), "drop EDA-redundant (covis_mean, covis_wsum)": (s_eda_redundant, 2),
             "L1 (C = 0.02)": (s_l1, 3), "stability selection (10 L1 fits, frequency >= 0.6)": (s_stability, 4), "tree importance >= 2% of gain": (s_tree_importance, 3),
             "permutation backward elimination": (s_perm_backward, 5), "recursive elimination to 8 (LightGBM gain)": (s_rfe, 5)}
final_extras = tuple(FAMILIES[f][0] for f in keepers)
def run1b():
    rows = []
    for sname, (fn, _) in SELECTORS.items():
        for bname, spec in BASE.items():
            r, _ = cv(replace(spec, name=f"{bname} | {sname}", extras=final_extras, selector=selector(fn)), dev10, seeds=SEEDS2)
            rows += r.with_columns(pl.lit(bname).alias("base_model"), pl.lit(sname).alias("selection")).to_dicts()
        print(f"  {sname}")
    return pl.DataFrame(rows)
cv1b = cached("step1_selection_rows", run1b)
sel1 = (cv1b.group_by("selection", "base_model", maintain_order=True).agg(pl.col("ndcg10").mean(), pl.col("ndcg10").std().alias("ndcg10_sd"), pl.col("pr_auc").mean(), pl.col("n_features").mean().alias("n_features")))
best_by_model = sel1.group_by("base_model").agg(pl.col("ndcg10").max().alias("best"))
sel1 = sel1.join(best_by_model, on="base_model").with_columns((pl.col("best") - pl.col("ndcg10")).alias("gap_to_best")).with_columns((pl.col("gap_to_best") <= NOISE).alias("within_noise"))
save_table("step1_selection", sel1); print(sel1.select("selection", "base_model", "ndcg10", "gap_to_best", "n_features", "within_noise").sort("base_model", "n_features"))

In [ ]:
# choose the smallest set within noise of the best, per model family; a lower-complexity tie-break follows the guide
chosen = {}
for bname in BASE:
    s = sel1.filter((pl.col("base_model") == bname) & pl.col("within_noise")).sort("n_features", "gap_to_best"); chosen[bname] = s.row(0, named=True)
    print(f"{bname}: smallest set within noise = '{chosen[bname]['selection']}' with {chosen[bname]['n_features']:.0f} features (NDCG@10 {chosen[bname]['ndcg10']:.4f}, gap to best {chosen[bname]['gap_to_best']:+.4f})")
allkeep = {bname: sel1.filter((pl.col("base_model") == bname) & (pl.col("selection") == "keep all")).row(0, named=True) for bname in BASE}
print({k: round(v["ndcg10"], 4) for k, v in allkeep.items()}, "keep-all NDCG@10 | reference", round(REFERENCE, 4))

In [ ]:
# which features do the chosen selections keep? frequency over the 6 training folds, and the set on the whole screening sample
from collections import Counter


def freq_and_full(fn, fam):
    cnt = Counter()
    for seed in SEEDS2:
        for tr, _ in folds(dev10.nq, seed):
            q_tr = dev10.take(tr); pr = Prep(fam).fit(q_tr); cnt.update(fn(pr.transform(q_tr), q_tr, None, pr.names, fam))
    pr = Prep(fam).fit(dev10); full = sorted(fn(pr.transform(dev10), dev10, None, pr.names, fam), key=F15.index)
    return cnt, full
cnt_l, full_l = freq_and_full(s_stability, "linear"); cnt_g, full_g = freq_and_full(s_rfe, "gbm")
sel_freq = pl.DataFrame({"feature": F15, "linear_stability_folds_of_6": [cnt_l.get(f, 0) for f in F15], "lightgbm_rfe_folds_of_6": [cnt_g.get(f, 0) for f in F15]}).sort("linear_stability_folds_of_6", "lightgbm_rfe_folds_of_6", descending=True)
save_table("step1_selected_features", sel_freq); print(sel_freq)
print("linear, stability selection on the whole screening sample:", full_l); print("LightGBM, recursive elimination on the whole screening sample:", full_g)


In [ ]:
# confirm on fresh fold seeds (44, 45), which were not used to choose the sets: which list is the smallest within noise for BOTH models?
LISTS = {"all 15 EDA features": F15, f"stability-selection list ({len(full_l)})": full_l, f"LightGBM recursive-elimination list ({len(full_g)})": full_g,
         "union of both lists": sorted(set(full_l) | set(full_g), key=F15.index)}
SEEDS_FRESH = (44, 45)
def run1c():
    rows = []
    for lname, cols in LISTS.items():
        for bname, spec in BASE.items():
            r, _ = cv(replace(spec, name=f"{bname} | {lname}", cols=tuple(cols)), dev10, seeds=SEEDS_FRESH)
            rows += r.with_columns(pl.lit(bname).alias("base_model"), pl.lit(lname).alias("feature_list"), pl.lit(len(cols)).alias("n_base_features")).to_dicts()
    return pl.DataFrame(rows)
cv1c = cached("step1_confirm_rows", run1c)
conf = cv1c.group_by("feature_list", "base_model", "n_base_features", maintain_order=True).agg(pl.col("ndcg10").mean(), pl.col("ndcg10").std().alias("ndcg10_sd"), pl.col("pr_auc").mean())
conf = conf.join(conf.group_by("base_model").agg(pl.col("ndcg10").max().alias("best")), on="base_model").with_columns((pl.col("best") - pl.col("ndcg10")).alias("gap_to_best"))
worst = conf.group_by("feature_list", "n_base_features").agg(pl.col("gap_to_best").max().alias("worst_gap"), pl.col("ndcg10").min().alias("worst_ndcg10")).sort("n_base_features")
save_table("step1_confirm", conf); save_table("step1_confirm_worst_gap", worst); print(conf.select("feature_list", "base_model", "ndcg10", "gap_to_best").sort("base_model", "feature_list")); print(worst)
pick = worst.filter(pl.col("worst_gap") <= NOISE).sort("n_base_features").row(0, named=True)
FEATURES = LISTS[pick["feature_list"]]
print(f"chosen list: {pick['feature_list']} = {FEATURES}; worst gap to the best over both models {pick['worst_gap']:.4f} (noise {NOISE:.4f})")
import json as _json

(ROOT / "modeling" / "artifacts" / "feature_list.json").write_text(_json.dumps({"features": FEATURES, "extras": [FAMILIES[f][0].name for f in keepers], "rule": "step 1, confirmed on seeds 44-45"}, indent=1))

In [ ]:
desc = {}
exp_rows = []
for r in fam1.iter_rows(named=True):
    exp_rows.append({"id": f"S1-fam-{r['family'][:28]}-{r['base_model'][:8]}".replace(" ", "_").replace(",", ""), "step": "1", "description": f"add family '{r['family']}' to F15", "features": f"F15 + {r['family']}", "model": r["base_model"], "parameters": "defaults",
                     "seeds": str(list(SEEDS2)), "ndcg10_mean": r["ndcg10"], "ndcg10_sd": r["paired_sd"], "pr_auc_mean": None, "pr_auc_sd": None, "fit_seconds": r["fit_seconds"], "notes": f"delta vs F15 {r['delta_ndcg10']:+.4f}; {r['verdict']}"})
for r in sel1.iter_rows(named=True):
    exp_rows.append({"id": f"S1-sel-{r['selection'][:24]}-{r['base_model'][:8]}".replace(" ", "_").replace(",", ""), "step": "1", "description": f"selection option '{r['selection']}' on {CURRENT}", "features": f"{r['n_features']:.0f} features", "model": r["base_model"], "parameters": "defaults",
                     "seeds": str(list(SEEDS2)), "ndcg10_mean": r["ndcg10"], "ndcg10_sd": r["ndcg10_sd"], "pr_auc_mean": r["pr_auc"], "notes": f"gap to best {r['gap_to_best']:.4f}; within noise {r['within_noise']}"})
print(log_experiments(exp_rows), "experiment rows in total")
fam_txt = "; ".join(f"{r['family']} ({r['base_model'].split()[0]}) {r['delta_ndcg10']:+.4f}" for r in fam1.sort("delta_ndcg10", descending=True).head(6).iter_rows(named=True))
sel_txt = "; ".join(f"{bn.split()[0]}: {c['selection']} ({c['n_features']:.0f} features, {c['ndcg10']:.4f})" for bn, c in chosen.items())
e1 = log_step(1, "Features", "Selection",
    "Eight families added one at a time to the 15 EDA features (ratios/interactions, bins, query-context aggregates, time parts, category frequency, cross-fitted category target rate, structure, missingness), each scored with the linear model and LightGBM on two fold seeds x 3 folds; nine selection options (keep all, near-constant, EDA no-signal, EDA-redundant, L1, stability selection, tree importance, permutation backward elimination, recursive elimination). Human-suggested features: none received.",
    f"Family effects on NDCG@10 against the same model without it (largest six of {fam1.height}): {fam_txt}. Noise threshold {NOISE:.4f}. Families kept by the rule: {', '.join(keepers) or 'none'}. Selection on {CURRENT}: keep-all NDCG@10 " + ", ".join(f"{k.split()[0]} {v['ndcg10']:.4f}" for k, v in allkeep.items()) + f" (reference {REFERENCE:.4f}). Smallest set within noise of the best: {sel_txt}.",
    f"Feature set going forward: {CURRENT}" + ("" if keepers else " (no family beat the noise threshold)") + f", reduced to the {len(FEATURES)}-feature list {', '.join(FEATURES)} ({pick['feature_list']}), the smallest of four candidate lists (all 15, the stability-selection list, the LightGBM recursive-elimination list, their union) whose gap to the best is within the noise threshold for both models on fresh seeds 44-45 (worst gap {pick['worst_gap']:.4f}).",
    "Guide rules: keep a family only if it helps at least one model beyond the noise and does not hurt the other; choose the smallest set within noise of the best; hypotheses were stated before testing.",
    "Families not kept: " + "; ".join(r["family"] for r in keep_rule.filter(~pl.col("family").is_in(keepers)).iter_rows(named=True)) + ". Selection options that lost more than the noise threshold to the best were not chosen (see the table).",
    "Medium: two fold seeds; every effect is compared with the guide's noise threshold, which is about four times wider than the paired fold-to-fold spread of the differences.",
    "Step 2 uses the selected feature list (modeling/artifacts/feature_list.json) for the whole model slate. Step 6 re-tests it by feature-group ablation and flags the query-category features.")
print(e1)

#### Chosen approach and rationale
**Decision.** Feature set going forward: F15 (no family beat the noise threshold), reduced to the 4-feature list rank_covis, order_rate, pop_cat_pct, covis_wsum (stability-selection list (4)), the smallest of four candidate lists (all 15, the stability-selection list, the LightGBM recursive-elimination list, their union) whose gap to the best is within the noise threshold for both models on fresh seeds 44-45 (worst gap 0.0028).

**Why.** Guide rules: keep a family only if it helps at least one model beyond the noise and does not hurt the other; choose the smallest set within noise of the best; hypotheses were stated before testing.

**Rejected.** Families not kept: query-context aggregates; time features (hour, weekday); non-linear forms (bins); missingness (count, joint pattern); categorical: cross-fitted category target rate; categorical: category frequency; structure (cluster distances, principal components); ratios, differences, interactions. Selection options that lost more than the noise threshold to the best were not chosen (see the table).

**Confidence.** Medium: two fold seeds; every effect is compared with the guide's noise threshold, which is about four times wider than the paired fold-to-fold spread of the differences.

#### Interpretation: what we understand from the results
No family clears the noise threshold. The largest nominal gains are small: time features +0.0017 for the linear model (5 of 6 folds better, but there are only three days of sessions, so hour and weekday can hardly generalise), query-context aggregates +0.0016 for LightGBM, structure features +0.0014 and +0.0009, ratios +0.0008; cross-fitted category target rate is slightly negative. This agrees with the EDA (category encodings add nothing) and says that the table's columns already hold what a pool-level rearrangement can add: the model is limited by the information in the candidate table, not by how it is written. Selection is where the choice matters. With the noise threshold at 0.0082 every selection option is a tie for both models, so the guide's rule picks the smallest set. On fresh seeds the 4-feature list (`rank_covis`, `order_rate`, `pop_cat_pct`, `covis_wsum`) scores 0.1783 (linear) and 0.1777 (LightGBM), against 0.1802 and 0.1799 for all 15 features and 0.1810 and 0.1804 for LightGBM's own 7-feature list: a consistent but small advantage of the larger sets (0.002 to 0.003), below the noise threshold. Two things stand out: the selected features are the ones the EDA already showed to matter (co-visitation and category popularity), and no feature set moves any learned model above the reference (0.1815).

#### Impact on next steps
Step 2 uses the selected feature list (modeling/artifacts/feature_list.json) for the whole model slate. Step 6 re-tests it by feature-group ablation and flags the query-category features.

#### Out of scope
- Features that need information not in the candidate table: similarity between the candidate and the prefix items (item embeddings exist in data/categories), the category of the last prefix item, and the covis score of the last item alone. They require rebuilding the candidate table from the raw prefix (issue #22 code), which is a data-phase change; raised in the summary as the most promising direction.
- A version of `hours_since_last_seen` measured from the session start (EDA issue #103).
- Human-suggested features: none were given.

## Step 2. Model slate
*Kind: selection. Issue #113.* Every listed model family at defaults, on the features chosen in Step 1, on the same folds (screening sample, seeds 42 and 43 x 3 folds). Heavy models are fit on at most 400,000 training rows (guide rule 9: screen on a sample). Fit time, per-query prediction time and size are recorded and checked against the assumed constraints (100 ms per query, 50 MB).

In [ ]:

from catboost import CatBoostClassifier
from model_cv import Spec
from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

FEATURES = json.loads((ROOT / "modeling" / "artifacts" / "feature_list.json").read_text())["features"]
ref0 = pl.read_csv(ROOT / "modeling" / "tables" / "step0_reference.csv").row(0, named=True)
NOISE, REFERENCE = ref0["noise_threshold"], ref0["ndcg10"]
print("features:", FEATURES, "| noise threshold", round(NOISE, 4), "| reference", round(REFERENCE, 4))

class Capped:
    """Fit on at most n random training rows (guide rule 9); scores through predict_proba or a squashed decision function."""
    def __init__(self, est, n=400_000, seed=42): self.est, self.n, self.seed = est, n, seed
    def fit(self, x, y):
        idx = np.random.default_rng(self.seed).choice(len(x), min(len(x), self.n), replace=False); self.est.fit(x[idx], y[idx]); return self
    def predict_proba(self, x):
        if hasattr(self.est, "predict_proba"): return self.est.predict_proba(x)
        s = 1 / (1 + np.exp(-self.est.decision_function(x))); return np.stack([1 - s, s], axis=1)

def mk(name, prep_family, factory, ladder, group):
    return name, (Spec(name, prep_family, factory, ladder, False, tuple(FEATURES)), group)
SLATE = dict([
    mk("regularised linear", "linear", lambda: LogisticRegression(C=1.0, max_iter=300), 1, "linear"),
    mk("linear SVM (linear kernel)", "linear", lambda: Capped(LinearSVC(C=0.1, class_weight="balanced", max_iter=2000)), 1, "linear"),
    mk("shallow tree (depth 3)", "linear", lambda: DecisionTreeClassifier(max_depth=3, min_samples_leaf=200, class_weight="balanced", random_state=SEED), 2, "single tree"),
    mk("deeper tree (depth 8)", "linear", lambda: DecisionTreeClassifier(max_depth=8, min_samples_leaf=200, class_weight="balanced", random_state=SEED), 2, "single tree"),
    mk("random forest", "linear", lambda: Capped(RandomForestClassifier(n_estimators=100, min_samples_leaf=50, class_weight="balanced_subsample", n_jobs=4, random_state=SEED)), 3, "bagged trees"),
    mk("extra trees", "linear", lambda: Capped(ExtraTreesClassifier(n_estimators=100, min_samples_leaf=50, class_weight="balanced_subsample", n_jobs=4, random_state=SEED)), 3, "bagged trees"),
    mk("histogram gradient boosting (scikit-learn)", "gbm", lambda: HistGradientBoostingClassifier(class_weight="balanced", random_state=SEED), 3, "boosting"),
    mk("XGBoost (hist)", "gbm", lambda: XGBClassifier(n_estimators=100, tree_method="hist", scale_pos_weight=200, n_jobs=4, random_state=SEED), 3, "boosting"),
    mk("LightGBM classifier (class weights)", "gbm", lambda: lgbm().make(), 3, "boosting"),
    mk("CatBoost (auto class weights)", "gbm", lambda: CatBoostClassifier(iterations=100, auto_class_weights="Balanced", thread_count=4, random_seed=SEED, verbose=0), 3, "boosting"),
    mk("small neural network (32-16, scikit-learn)", "linear", lambda: Capped(MLPClassifier((32, 16), early_stopping=True, max_iter=40, random_state=SEED), n=300_000), 6, "neural"),
])
SLATE["LightGBM lambdarank"] = (replace(lgbm(name="LightGBM lambdarank", rank=True), cols=tuple(FEATURES)), "boosting")
SKIPPED = {"RBF-kernel SVM": "training cost grows with the square of the rows (1.3 million per fold); the linear SVM is run instead",
           "explainable boosting machine": "interpretability was not required (assumed in Step 0) and the interpret package is not installed",
           "nearest neighbours": "small data only: run on a 3,000-query subsample next to the linear model (see below)"}
def run2():
    rows = []
    for name, (spec, grp) in SLATE.items():
        t0 = time.time(); r, _ = cv(replace(spec, name=name), dev10, seeds=SEEDS2); rows += r.with_columns(pl.lit(grp).alias("group")).to_dicts(); print(f"  {name}: {time.time() - t0:.0f} s")
    return pl.DataFrame(rows)
SEEDS2 = (42, 43)
cv2 = cached("step2_slate_rows", run2)

In [ ]:
# nearest neighbours on a 3,000-query subsample (small data only), against the linear model on the same rows
dev3 = load_dev(3000, seed=SEED)
def run2b():
    rows = []
    for name, spec in {"nearest neighbours (k = 100) on 3,000 queries": Spec("nn", "linear", lambda: KNeighborsClassifier(n_neighbors=100, n_jobs=4), 2, False, tuple(FEATURES)), "regularised linear on the same 3,000 queries": replace(SLATE["regularised linear"][0], name="lin3")}.items():
        r, _ = cv(replace(spec, name=name), dev3, seeds=(42,)); rows += r.to_dicts()
    return pl.DataFrame(rows)
cv2b = cached("step2_knn_rows", run2b); sum2b = summarise(cv2b); save_table("step2_knn", sum2b); print(sum2b.select("option", "ndcg10", "ndcg10_sd", "pr_auc", "fit_seconds", "predict_seconds", "size_mb"))

In [ ]:
sum2 = summarise(cv2).join(cv2.select("option", "group").unique(), on="option")
lat = cv2.group_by("option").agg((pl.col("predict_seconds") / pl.col("queries")).mean().mul(1000).alias("ms_per_query"))
sum2 = sum2.join(lat, on="option").with_columns((pl.col("ndcg10") - REFERENCE).alias("vs_reference"), ((pl.col("ms_per_query") > 100) | (pl.col("size_mb") > 50)).alias("breaks_constraint"))
paired2 = cv2.join(cv0.filter(pl.col("option") == "co-visitation rank").select("seed", "fold", pl.col("ndcg10").alias("ref")), on=["seed", "fold"]).with_columns((pl.col("ndcg10") - pl.col("ref")).alias("d")).group_by("option").agg(pl.col("d").std().alias("paired_sd_vs_reference"), (pl.col("d") > 0).sum().alias("folds_above_reference"))
sum2 = sum2.join(paired2, on="option"); save_table("step2_slate", sum2)
print(sum2.select("option", "ndcg10", "ndcg10_sd", "vs_reference", "folds_above_reference", "pr_auc", "fit_seconds", "ms_per_query", "size_mb", "breaks_constraint").sort("ndcg10", descending=True))
for k, v in SKIPPED.items(): print("skipped:", k, "->", v)

In [ ]:
# shortlist: everything within the noise threshold of the leader, at most five, at least two families; more than five -> the five lowest on the ladder
ok = sum2.filter(~pl.col("breaks_constraint")); leader = ok.sort("ndcg10", descending=True).row(0, named=True)
tied = ok.filter(pl.col("ndcg10") >= leader["ndcg10"] - NOISE)
ladder = {n: SLATE[n][0].ladder for n in SLATE}
tied = tied.with_columns(pl.col("option").replace_strict(ladder, default=9).alias("rung")).sort("rung", "ndcg10", descending=[False, True])
short = tied.head(5)["option"].to_list()
if short and tied.filter(pl.col("option").is_in(short))["group"].n_unique() < 2:
    extra = tied.filter(~pl.col("option").is_in(short) & (pl.col("group") != tied.filter(pl.col("option").is_in(short))["group"][0])); short = short[:4] + extra.head(1)["option"].to_list()
print(f"leader (within constraints): {leader['option']} {leader['ndcg10']:.4f}; {tied.height} of {ok.height} models are within the noise threshold {NOISE:.4f} of it")
print("shortlist:", short); print("families in the shortlist:", sorted(sum2.filter(pl.col("option").is_in(short))["group"].unique().to_list()))
above = sum2.filter(pl.col("vs_reference") > NOISE)["option"].to_list()
print("models that beat the reference (co-visitation rank) by more than the noise threshold:", above or "none")
print(f"best learned model minus reference: {sum2['ndcg10'].max() - REFERENCE:+.4f} (noise {NOISE:.4f}, success criterion {2 * NOISE:.4f})")

In [ ]:
exp2 = []
for r in sum2.iter_rows(named=True):
    exp2.append({"id": "S2-" + "".join(c if c.isalnum() else "_" for c in r["option"])[:50], "step": "2", "description": f"{r['option']} at defaults on the selected features", "features": ", ".join(FEATURES), "model": r["option"], "parameters": "defaults (heavy models fit on at most 400,000 rows)",
                 "seeds": str(list(SEEDS2)), "ndcg10_mean": r["ndcg10"], "ndcg10_sd": r["ndcg10_sd"], "pr_auc_mean": r["pr_auc"], "pr_auc_sd": r["pr_auc_sd"], "fit_seconds": r["fit_seconds"], "predict_seconds": r["predict_seconds"], "size_mb": r["size_mb"],
                 "notes": f"{r['ms_per_query']:.2f} ms per query; vs reference {r['vs_reference']:+.4f}; constraint break {r['breaks_constraint']}"})
log_experiments(exp2)
tbl = "; ".join(f"{r['option']} {r['ndcg10']:.4f}" for r in sum2.sort("ndcg10", descending=True).iter_rows(named=True))
e2 = log_step(2, "Model slate", "Selection",
    "Run at defaults on the selected features (" + ", ".join(FEATURES) + "), same folds, seeds 42-43 x 3 folds: regularised linear, linear SVM, shallow and deeper decision tree, random forest, extra trees, histogram gradient boosting, XGBoost, LightGBM (classifier and lambdarank), CatBoost, small neural network (scikit-learn; PyTorch not used because of the OpenMP conflict with LightGBM), nearest neighbours (3,000-query subsample). Skipped with reasons: " + "; ".join(f"{k} ({v})" for k, v in SKIPPED.items() if not k.startswith("nearest")) + ".",
    f"NDCG@10 (mean over 6 folds): {tbl}. Reference (co-visitation rank) {REFERENCE:.4f}; noise threshold {NOISE:.4f}. Best learned model minus reference {sum2['ndcg10'].max() - REFERENCE:+.4f}; models above the reference by more than the noise: {', '.join(above) or 'none'}. Constraint breakers (over 100 ms per query or 50 MB): {', '.join(sum2.filter(pl.col('breaks_constraint'))['option'].to_list()) or 'none'}. Nearest neighbours on 3,000 queries: " + "; ".join(f"{r['option']} {r['ndcg10']:.4f}" for r in sum2b.iter_rows(named=True)) + f". {tied.height} of {ok.height} models are within the noise threshold of the leader ({leader['option']}).",
    "Shortlist for tuning (everything within noise of the leader, at most five, at least two families; over five ties keep the lowest on the ladder): " + ", ".join(short) + ".",
    "Guide rules: run every listed option at defaults on shared folds; remove constraint breakers; shortlist within noise, at most five, at least two families, ladder as tie-break.",
    "Not shortlisted: models outside the noise threshold of the leader and, among the ties, the ones higher on the ladder. Skipped: " + "; ".join(SKIPPED) + ".",
    "Medium: two fold seeds; the noise threshold is wide, so most models tie and the shortlist follows the ladder (see the paired columns for which differences are consistent).",
    "Step 3 tunes the shortlist with one shared budget. If nothing beats the reference by more than the noise threshold after Step 3, the guide requires stopping to ask.")
print(e2)